# Bootstrap: Monte Carlo na datech

*Nad rámec skripta — aplikace.*

---

Všechny ostatní sešity simulují **model**, který jsme si sami zadali: známe rozdělení, generujeme z něj, počítáme střední hodnotu. Bootstrap dělá něco jiného.

Máte **naměřená data** a spočítáte z nich nějakou statistiku — průměr, medián, korelaci, poměr kvantilů. Otázka zní: *jak přesné to číslo je?* Pro průměr existuje vzorec $\hat\sigma/\sqrt n$. Pro medián? Pro poměr dvou kvantilů? Pro korelaci? Vzorce buď neexistují, nebo platí jen asymptoticky a za předpokladů, které nikdo neověřuje.

**Bootstrap** (Efron, 1979) odpovídá bez vzorců. Myšlenka je drze jednoduchá:

> Neznáme skutečné rozdělení, ale máme vzorek. Tak **tvařme se, že vzorek JE to rozdělení**, a losujme z něj.

Konkrétně: z $n$ naměřených hodnot losujeme $n$ hodnot **s vracením**, spočítáme statistiku, a opakujeme tisíckrát. Rozptyl těch tisíců hodnot odhadne rozptyl té původní.

Uvidíte:

- bootstrapový interval pro průměr a srovnání s klasickým vzorcem
- medián a korelaci, kde vzorec není
- **experiment s pokrytím** — drží bootstrap, co slibuje?
- a kde selhává (protože selhává)

In [ ]:
import math
import random

import matplotlib.pyplot as plt

random.seed(1)  # opakovatelné výsledky


def prumer(d):
    return sum(d) / len(d)


def median(d):
    s = sorted(d)
    n = len(s)
    return s[n // 2] if n % 2 else (s[n // 2 - 1] + s[n // 2]) / 2


def smerodatna_odchylka(d):
    m = prumer(d)
    return math.sqrt(sum((x - m) ** 2 for x in d) / (len(d) - 1))


def bootstrap(data, statistika, B=4000):
    """B bootstrapových replikací statistiky."""
    n = len(data)
    return [statistika([data[random.randrange(n)] for _ in range(n)]) for _ in range(B)]


def interval(replikace, alpha=0.05):
    """Percentilový interval spolehlivosti."""
    s = sorted(replikace)
    B = len(s)
    return s[int(alpha / 2 * B)], s[int((1 - alpha / 2) * B)]


# "naměřená" data: doby odezvy serveru v ms — zešikmená, jak to u čekacích dob bývá
SKUTECNY_LOG_STRED, SKUTECNY_LOG_SD = 1.0, 0.6
SKUTECNY_PRUMER = math.exp(SKUTECNY_LOG_STRED + SKUTECNY_LOG_SD**2 / 2)
SKUTECNY_MEDIAN = math.exp(SKUTECNY_LOG_STRED)


def nova_mereni(n):
    return [math.exp(random.gauss(SKUTECNY_LOG_STRED, SKUTECNY_LOG_SD)) for _ in range(n)]


data = nova_mereni(40)
print(f"naměřeno {len(data)} hodnot, prvních osm: {[round(x, 2) for x in data[:8]]}")
print(f"průměr vzorku {prumer(data):.4f}, medián {median(data):.4f}")
print(f"\n(pro kontrolu — data jsme vyrobili z lognormálního rozdělení,")
print(f" takže skutečný průměr je {SKUTECNY_PRUMER:.4f} a medián {SKUTECNY_MEDIAN:.4f};")
print(f" v praxi bychom to pochopitelně nevěděli)")

## 1. Interval pro průměr: srovnání se vzorcem

Začneme tam, kde vzorec **máme**, abychom bootstrapu mohli věřit jinde. Klasický interval pro střední hodnotu je

$$\bar x \pm 1.96\,\frac{\hat\sigma}{\sqrt n}.$$

Bootstrap k témuž číslu dojde úplně jinou cestou — losováním z vlastních dat.

In [ ]:
rep_prumer = bootstrap(data, prumer)
lo_b, hi_b = interval(rep_prumer)

m = prumer(data)
polo = 1.96 * smerodatna_odchylka(data) / math.sqrt(len(data))
print(f"{'metoda':>24s} {'interval':>26s} {'šířka':>8s}")
print(f"{'klasický (CLV)':>24s} [{m - polo:>10.4f}, {m + polo:>10.4f}] {2 * polo:>8.4f}")
print(f"{'bootstrap':>24s} [{lo_b:>10.4f}, {hi_b:>10.4f}] {hi_b - lo_b:>8.4f}")
print(f"\nbootstrapová směrodatná chyba: {smerodatna_odchylka(rep_prumer):.4f}")
print(f"klasická  sigma/sqrt(n):       {smerodatna_odchylka(data) / math.sqrt(len(data)):.4f}")

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.2))
ax = axes[0]
ax.hist(data, bins=20, alpha=0.65, color="tab:blue")
ax.axvline(m, color="tab:red", ls="--", label=f"průměr vzorku {m:.2f}")
ax.set_xlabel("doba odezvy [ms]")
ax.set_ylabel("počet")
ax.set_title(f"Naše data ({len(data)} hodnot)", fontsize=10)
ax.legend(fontsize=8)

ax = axes[1]
ax.hist(rep_prumer, bins=50, alpha=0.65, color="tab:green")
ax.axvline(lo_b, color="k", ls="--")
ax.axvline(hi_b, color="k", ls="--", label="bootstrapový 95% interval")
ax.axvline(m - polo, color="tab:red", ls=":")
ax.axvline(m + polo, color="tab:red", ls=":", label="klasický 95% interval")
ax.set_xlabel("průměr bootstrapového vzorku")
ax.set_ylabel("počet replikací")
ax.set_title("Rozdělení průměru podle bootstrapu", fontsize=10)
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

Oba intervaly vyjdou prakticky stejně — což je přesně to, co chceme vidět. Bootstrap nevyrobil nic nového tam, kde vzorec funguje; jen ho potvrdil bez toho, aby ho znal.

Teď to zajímavější.

## 2. Kde vzorec není

**Medián.** Asymptotický vzorec existuje, ale potřebuje odhad hustoty v mediánu — což je samo o sobě úloha. Bootstrap nepotřebuje nic.

**Korelace, poměry kvantilů, rozdíl mediánů dvou skupin.** Tam už vzorce buď nejsou, nebo jsou tak nepohodlné, že je nikdo nepoužívá.

Bootstrap s nimi zachází úplně stejně jako s průměrem — **změní se jediná funkce**.

In [ ]:
def pomer_kvantilu(d):
    """95. percentil děleno mediánem — míra 'jak zlý je chvost'."""
    s = sorted(d)
    return s[int(0.95 * len(s))] / median(s)


print(f"{'statistika':>22s} {'hodnota':>10s} {'95% interval (bootstrap)':>28s}")
for nazev, stat in (("průměr", prumer),
                    ("medián", median),
                    ("směrodatná odchylka", smerodatna_odchylka),
                    ("95. percentil / medián", pomer_kvantilu)):
    rep = bootstrap(data, stat)
    lo, hi = interval(rep)
    print(f"{nazev:>22s} {stat(data):>10.4f} [{lo:>11.4f}, {hi:>11.4f}]")

print("\nVšechny čtyři řádky vznikly týmž kódem — lišila se jen předaná funkce.")
print("To je na bootstrapu to hlavní: nepotřebuje pro každou statistiku novou teorii.")

## 3. Drží bootstrap, co slibuje?

Interval se nazývá 95%, takže by měl skutečnou hodnotu pokrývat v 95 % případů. Ověříme to stejně jako u pojišťovny ve cvičení 9: vygenerujeme mnoho **nových** datových souborů, pro každý spočítáme interval a změříme, jak často trefí.

Tady si to můžeme dovolit, protože jsme data sami vyrobili a skutečnou hodnotu známe.

In [ ]:
def pokryti(n, stat, skutecna, opakovani=400, B=600):
    uspech_boot = 0
    uspech_clv = 0
    for _ in range(opakovani):
        d = nova_mereni(n)
        lo, hi = interval(bootstrap(d, stat, B))
        uspech_boot += lo <= skutecna <= hi
        if stat is prumer:
            m_, p_ = prumer(d), 1.96 * smerodatna_odchylka(d) / math.sqrt(n)
            uspech_clv += m_ - p_ <= skutecna <= m_ + p_
    return uspech_boot / opakovani, (uspech_clv / opakovani if stat is prumer else None)


print(f"{'n':>5s} {'statistika':>10s} {'pokrytí bootstrapu':>20s} {'pokrytí CLV':>13s}")
for n in (20, 40, 100, 400):
    pb, pc = pokryti(n, prumer, SKUTECNY_PRUMER)
    print(f"{n:>5d} {'průměr':>10s} {pb:>20.3f} {pc:>13.3f}")
    pb, _ = pokryti(n, median, SKUTECNY_MEDIAN)
    print(f"{n:>5d} {'medián':>10s} {pb:>20.3f} {'—':>13s}")

print("\nPro malé n obě metody slibují víc, než plní (kolem 90 % místo 95 %) — data jsou")
print("zešikmená a čtyřicet hodnot na to nestačí. S rostoucím n se obě blíží k 95 %.")
print("Bootstrap tedy NENÍ kouzlo: dědí tutéž slabinu jako klasický vzorec.")

Tohle je důležitější než samotná metoda. Bootstrap se často prodává jako řešení, které funguje vždycky — a experiment ukazuje, že **u malých zešikmených vzorků slibuje přesnost, kterou nemá**, úplně stejně jako klasický vzorec. Nedodá informaci, která v datech není.

## 4. Kde bootstrap selže doopravdy

Jeden případ stojí za ukázku, protože je názorný: **maximum**.

Bootstrapový vzorek se losuje z naměřených hodnot, takže nikdy nemůže obsahovat nic většího než pozorované maximum. Rozdělení bootstrapovaného maxima je proto useknuté shora — a interval pro skutečné maximum je systematicky špatně.

In [ ]:
data_velka = nova_mereni(200)
skutecne_max_rozdeleni = math.exp(SKUTECNY_LOG_STRED + SKUTECNY_LOG_SD * 2.5)  # jen pro měřítko

rep_max = bootstrap(data_velka, max, 4000)
print(f"pozorované maximum: {max(data_velka):.4f}")
print(f"bootstrapové replikace maxima: {len(set(rep_max))} různých hodnot ze 4000 replikací")
print(f"podíl replikací rovných pozorovanému maximu: "
      f"{sum(1 for v in rep_max if v == max(data_velka)) / len(rep_max):.1%}")

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.2))
ax = axes[0]
ax.hist(bootstrap(data_velka, median, 4000), bins=40, alpha=0.7, color="tab:green")
ax.set_xlabel("medián bootstrapového vzorku")
ax.set_ylabel("počet")
ax.set_title("Medián: pěkné spojité rozdělení ✓", fontsize=10)

ax = axes[1]
ax.hist(rep_max, bins=40, alpha=0.7, color="tab:red")
ax.axvline(max(data_velka), color="k", ls="--", label="pozorované maximum")
ax.set_xlabel("maximum bootstrapového vzorku")
ax.set_ylabel("počet")
ax.set_title("Maximum: pár diskrétních hodnot, shora useknuté ✗", fontsize=10)
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

print("\nBootstrap funguje pro statistiky, které se mění HLADCE s rozdělením dat")
print("(průměr, medián, kvantily uvnitř, korelace). Pro ty, které závisí na")
print("samotném okraji (maximum, minimum, nosič), selhává — a žádný počet")
print("replikací to nespraví, protože informace o chvostu v datech prostě není.")

## Shrnutí

| Pojem | Co si pamatovat |
|-------|-----------------|
| Myšlenka | tvař se, že vzorek **je** rozdělení, a losuj z něj s vracením |
| Co umí | interval pro libovolnou statistiku bez odvozování vzorce |
| Kontrola | pro průměr dá prakticky totéž co klasický $\hat\sigma/\sqrt n$ |
| Univerzálnost | změní se **jediná funkce**: medián, korelace, poměr kvantilů… |
| Pokrytí | u malých zešikmených vzorků kolem 90 %, ne 95 % — stejně jako vzorec |
| Kde selže | statistiky závislé na okraji (maximum, minimum) — vzorek nedosáhne dál, než dosáhl |
| Čím se liší od zbytku | simuluje se z **dat**, ne z modelu, který jsme si zadali |

Hlavní myšlenka: bootstrap nedodá informaci, kterou data neobsahují. Jen vytěží tu, která tam je, bez nutnosti odvozovat vzorec pro každou statistiku zvlášť.

### Zkuste si

1. Spočítejte bootstrapový interval pro **rozdíl mediánů** dvou skupin a porovnejte ho s klasickým t-testem na průměrech. Kdy se liší závěr?
2. Zkuste **vychýlení**: bootstrapový odhad vychýlení je $\overline{\theta^*} - \hat\theta$. Je směrodatná odchylka vychýlená? O kolik?
3. Udělejte experiment s pokrytím pro `pomer_kvantilu`. Drží interval lépe, nebo hůř než u mediánu?
4. Místo percentilového intervalu zkuste **BCa** nebo aspoň základní („basic") interval $[2\hat\theta - q_{0.975},\ 2\hat\theta - q_{0.025}]$. Zlepší se pokrytí u zešikmených dat?
5. Pro $n = 5$ spočítejte, kolik různých bootstrapových vzorků vůbec existuje. Co to říká o použitelnosti metody u velmi malých dat?
6. Nahraďte lognormální data normálními a zopakujte experiment s pokrytím. Potvrdí se, že problém dělá zešikmení, a ne bootstrap sám?